In [15]:
import torch
import numpy as np


In [16]:
# Step 1: 生成旋转频率
def get_rotary_freqs(dim:int,seq_len:int,theta:float = 100000.0):
    """
    生成 RoPE 的旋转频率(angle_{m,i}=m⋅θi)

    Args:
        dim: 嵌入维度（必须是偶数）
        seq_len: 序列长度
        theta: 基础频率参数

    Returns:
        freqs: shape (seq_len, dim // 2)，每个位置每个维度对的频率
    """
    # 计算每个维度对的基础频率
    # theta_i = 10000^(-2i/d)，i = 0, 1, ..., d/2-1
    i = torch.arange(0,dim//2,dtype=torch.float32)
    freqs = theta ** (-2 * i / dim) # shape : (dim//2,)
    
    pos = torch.arange(seq_len,dtype=torch.float32) # shape: (seq_len,)
    
    # 计算每个位置的角度：position * frequency
    # 外积得到 (seq_len, dim // 2) 的矩阵,angles[i,j]=pos[i]⋅freqs[j]
    angles = torch.outer(pos,freqs)
    
    return angles

# # 测试
# dim = 64
# seq_len = 128
# angles = get_rotary_freqs(dim, seq_len)
# print(f"Angles shape: {angles.shape}")  # (128, 32)
# print(f"Angles[0]: {angles[0][:5]}")    # 位置 0 的前 5 个维度对的角度
# print(f"Angles[1]: {angles[1][:5]}")    # 位置 1 的前 5 个维度对的角度

In [17]:
# Step 2: 构建 sin/cos 缓存
def get_rotary_embedding(dim : int,seq_len:int,theta:float = 100000.0):
    """
    预计算 RoPE 的 sin 和 cos 值

    Returns:
        cos: shape (seq_len, dim)
        sin: shape (seq_len, dim)
    """
    angles = get_rotary_freqs(dim, seq_len, theta)
    
    # 计算 cos 和 sin
    cos = torch.cos(angles)
    sin = torch.sin(angles)
    
    # 将 (seq_len, dim//2) 扩展为 (seq_len, dim)，与 rotate_half 配合使用
    # 工程上为了简洁使用前半段与后半段x配对(x0,x_dim/2),而不是x0,x1配对
    cos = torch.cat([cos, cos], dim=-1)
    sin = torch.cat([sin, sin], dim=-1)
    
    return cos,sin

# # 测试
# cos, sin = get_rotary_embedding(dim=64, seq_len=128)
# print(f"Cos shape: {cos.shape}")  # (128, 64)
# print(f"Sin shape: {sin.shape}")  # (128, 64)

In [18]:
# Step 3: 应用旋转变换
def rotate_half(x):
    """
    这里实现的是前半段与后半段配对，与相邻配对效果一样
    将向量的前半部分和后半部分交换，并对后半部分取负
    [x1, x2, x3, x4] -> [-x3, -x4, x1, x2]

    这是实现旋转的关键辅助函数
    """
    x1 = x[..., : x.shape[-1] // 2] # 最后一维(head_dim)取前半部分
    x2 = x[..., x.shape[-1] // 2 :] # 最后一维取后半部分
    
    return torch.cat((-x2, x1), dim=-1)

def apply_rotary_pos_emb(q, k, cos, sin):
    """
    应用 RoPE 旋转变换（LLaMA 风格实现）

    Args:
        q: Query，shape (batch, seq_len, num_heads, head_dim)
        k: Key，shape (batch, seq_len, num_heads, head_dim)
        cos: shape (seq_len, head_dim)
        sin: shape (seq_len, head_dim)

    Returns:
        q_rot, k_rot: 旋转后的 Query 和 Key

    旋转公式：
        q' = q * cos + rotate_half(q) * sin
        k' = k * cos + rotate_half(k) * sin
        
    原向量 * cos + rotate_half(原向量) * sin
    = [x, y] * cos + [-y, x] * sin
    = [x*cos - y*sin, y*cos + x*sin]
    """
    # 调整 cos/sin 形状以便广播: (seq_len, head_dim) -> (1, seq_len, 1, head_dim)
    cos = cos.unsqueeze(0).unsqueeze(2)
    sin = sin.unsqueeze(0).unsqueeze(2)

    # 应用旋转
    q_embed = (q * cos) + (rotate_half(q) * sin)
    k_embed = (k * cos) + (rotate_half(k) * sin)

    return q_embed, k_embed

In [ ]:
class RotaryPositionEmbedding(torch.nn.Module):
    def __init__(self,dim:int,max_seq_len:int = 32768,theta:float = 100000.0):
        """
        Args:
            dim: 每个注意力头的维度
            max_seq_len: 最大序列长度
            theta: 基础频率参数
        """
        super().__init__()
        self.dim = dim
        self.max_seq_len = max_seq_len
        self.theta = theta
        
        cos,sin = get_rotary_embedding(dim,max_seq_len,theta)
        self.register_buffer('cos', cos)
        self.register_buffer('sin', sin)
        
    def forward(self,q:torch.Tensor,k:torch.Tensor,pos:torch.Tensor=None):
        """
        对 Query 和 Key 应用 RoPE

        Args:
            q: Query，shape (batch, seq_len, num_heads, head_dim)
            k: Key，shape (batch, seq_len, num_heads, head_dim)
            positions: 位置索引，默认为 [0, 1, 2, ..., seq_len-1]

        Returns:
            q_rot, k_rot: 旋转后的 Query 和 Key
        """
        seq_len = q.shape[1]
        
        # 获取当前序列长度的 cos/sin
        cos = self.cos[:seq_len]
        sin = self.sin[:seq_len]
        
        # 应用旋转
        q_rot, k_rot = apply_rotary_pos_emb(q, k, cos, sin)

        return q_rot, k_rot


Q_rot shape: torch.Size([2, 128, 8, 64])
K_rot shape: torch.Size([2, 128, 8, 64])


In [27]:
def verify_relative_position_invariance():
    """
    验证 RoPE 的相对位置不变性
    """
    dim = 64
    max_seq_len = 100

    # 预计算 cos/sin
    cos, sin = get_rotary_embedding(dim, max_seq_len)

    # 创建两个相同的向量
    torch.manual_seed(42)
    q = torch.randn(1, 1, 1, dim)
    k = torch.randn(1, 1, 1, dim)

    # 场景 1：q 在位置 0，k 在位置 5（相对位置 = 5）
    cos1_q, sin1_q = cos[0:1], sin[0:1]
    cos1_k, sin1_k = cos[5:6], sin[5:6]

    q1_rot, _ = apply_rotary_pos_emb(q, q, cos1_q, sin1_q)
    _, k1_rot = apply_rotary_pos_emb(k, k, cos1_k, sin1_k)

    dot_product_1 = (q1_rot * k1_rot).sum()

    # 场景 2：q 在位置 10，k 在位置 15（相对位置仍然是 5）
    cos2_q, sin2_q = cos[10:11], sin[10:11]
    cos2_k, sin2_k = cos[15:16], sin[15:16]

    q2_rot, _ = apply_rotary_pos_emb(q, q, cos2_q, sin2_q)
    _, k2_rot = apply_rotary_pos_emb(k, k, cos2_k, sin2_k)

    dot_product_2 = (q2_rot * k2_rot).sum()

    print(f"位置 (0, 5) 的内积: {dot_product_1.item():.6f}")
    print(f"位置 (10, 15) 的内积: {dot_product_2.item():.6f}")
    print(f"差异: {abs(dot_product_1.item() - dot_product_2.item()):.10f}")
    print("验证通过！" if abs(dot_product_1.item() - dot_product_2.item()) < 1e-5 else "验证失败！")


# verify_relative_position_invariance()


In [26]:
import matplotlib.pyplot as plt
import seaborn as sns

def visualize_rope_heatmap():
    """
    可视化 RoPE 编码的热力图
    """
    dim = 64
    seq_len = 128

    cos, sin = get_rotary_embedding(dim, seq_len)

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Cos 热力图
    sns.heatmap(cos.numpy()[:64, :], ax=axes[0], cmap='RdBu', center=0)
    axes[0].set_title('RoPE Cos Values')
    axes[0].set_xlabel('Dimension')
    axes[0].set_ylabel('Position')

    # Sin 热力图
    sns.heatmap(sin.numpy()[:64, :], ax=axes[1], cmap='RdBu', center=0)
    axes[1].set_title('RoPE Sin Values')
    axes[1].set_xlabel('Dimension')
    axes[1].set_ylabel('Position')

    plt.tight_layout()
    plt.savefig('rope_heatmap.png', dpi=150)
    plt.show()

    print("观察要点：")
    print("1. 低维度（左侧）变化快 -> 捕捉短距离依赖")
    print("2. 高维度（右侧）变化慢 -> 捕捉长距离依赖")
    print("3. 每个维度都是周期函数，频率不同")


# visualize_rope_heatmap()
